# NenuFAR Solar Imaging Workflow (DEMO for SOLER User Workshop 2)

This notebook demonstrates a **notebook-driven UI workflow (ipywidgets)** for NenuFAR solar interferometric imaging:

- **Step0 (optional):** scan / select candidate SBs
- **Step1–4:** MS preparation → ROI cut → calibration (DP3) → imaging (WSClean) → FITS
- **Step4 quicklook:** generate PNG quicklooks / optional movies from `*-image.fits`
- **Step5A/5B:** quiet-Sun based **ionospheric offset (WCS shift)** solve + apply (write corrected FITS)
- **Step5C:** centroid extraction (2D Gaussian fit within an ROI) + tables + optional movie

---

## Where to run / access
This workflow is intended to run on the **Nançay computing environment** (e.g., `nancep`), with access to NenuFAR data paths and the required compute permissions.
Access may require approval from the **NenuFAR KP11 (Solar) team**.

---

## Two external files (not in this repo)
Some Step1–4 actions require two external resources (paths below are Nançay-specific):

- Container image: `linc_latest.sif`
- Calibration DB: `CasA.sourcedb`

If you need access to them, please contact the author / KP11 Solar team.

---

## Output folders (important)
Each step writes outputs under a `step*_outputs_YYYYMMDD/` folder (event-date tag).
Step5 writes:
- `step_iocorrect_outputs_YYYYMMDD/` (Step5A/5B)
- `step_iocentroid_outputs_YYYYMMDD/` (Step5C)

Check the printed notebook logs for the exact output paths for your run.

In [5]:
import importlib
import nenufar_ui
import nenufar_sb_scan
import post_analysis_tools

importlib.reload(nenufar_sb_scan)
importlib.reload(nenufar_ui)
importlib.reload(post_analysis_tools)

from nenufar_ui import load_and_select_sb
from nenufar_ui import run_step1_ui
from nenufar_ui import run_step2_zoom_ui
from nenufar_ui import run_step3_calib_ui
from nenufar_ui import run_step4_wsclean_ui
from nenufar_ui import run_step4_quicklook_ui

from post_analysis_tools import run_iocorr_solve_ui
from post_analysis_tools import run_iocorr_apply_ui
from post_analysis_tools import run_iocorr_centroid_ui
from post_analysis_tools import run_beam_propagation_ui

In [ ]:
import sys
from pathlib import Path

# --------------------------------------------------
# Global base configuration
# --------------------------------------------------

# Set this to the local path of your cloned workflow repository
CODE_ROOT = Path("/path/to/your/repository")
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# Root path to the local NenuFAR data tree
BASE = Path("/databf/nenufar-nri/LT11")

# Set this to your own working directory for workflow outputs
WORKROOT = Path("/path/to/your/workflow_workspace")

print("CODE_ROOT =", CODE_ROOT)
print("BASE      =", BASE)
print("WORKROOT  =", WORKROOT)

# Scan & SB Selection

## Purpose
Locate SUN and CAS_A events for a given date, align SBs, and generate a processing plan (JSON).

## What it does
- Find SUN_TRACKING event
- Find all CAS_A_TRACKING candidates (pre & post)
- Match SBs by frequency
- Allow user SB + calibrator selection
- Save selected SB list to JSON

## Output
`selected_sb_pair_list.json`

Contains:
- selected_sb
- sun_ms
- casa_pre_ms
- casa_post_ms
- casa_ms (if chosen)

⚠ No DP3 processing is done here.
This step only builds the processing plan.

**How to use**
1. Set:
   - `BASE` (NenuFAR data root, e.g. `/databf/nenufar-nri/LT11`)
   - `WORKROOT` (your working directory for outputs)
2. Run the cell to open the widget UI.
3. Adjust Year/Month/Day, frequency range, search filters, then click **Run**.
4. The output table helps you identify SB(s) to process in Step1–4.

**Output**
- A selected SB plan JSON is typically written under a `...SUN_TRACKING/selected_sb_pair_list.json` folder (see the printed output).

In [ ]:
load_and_select_sb(str(BASE), str(WORKROOT))

In [ ]:
from pathlib import Path
import json

LATEST_PLAN_FILE = WORKROOT / "latest_selected_plan.txt"
if not LATEST_PLAN_FILE.exists():
    raise FileNotFoundError(f"Missing {LATEST_PLAN_FILE}. Please run load_and_select_sb(...) first.")

PLAN_JSON = Path(LATEST_PLAN_FILE.read_text().strip())
EVENT_ROOT = PLAN_JSON.parent
EVENT_NAME = EVENT_ROOT.name
EVENT_TAG = EVENT_NAME[:8]

STEP1_ROOT = WORKROOT / f"step1_outputs_{EVENT_TAG}"
STEP2_ROOT = WORKROOT / f"step2_outputs_{EVENT_TAG}"
STEP3_ROOT = WORKROOT / f"step3_outputs_{EVENT_TAG}"
STEP4_ROOT = WORKROOT / f"step4_outputs_{EVENT_TAG}"
IOCORR_ROOT = WORKROOT / f"step_iocorrect_outputs_{EVENT_TAG}"
IOCENTROID_ROOT = WORKROOT / f"step_iocentroid_outputs_{EVENT_TAG}"
BEAMPROP_ROOT = WORKROOT / f"beamprop_outputs_{EVENT_TAG}"

# --------------------------------------------------
# Local processing resources
# --------------------------------------------------

# Set this to the local path of the container used for the main imaging workflow
CONTAINER_PATH = CODE_ROOT / "linc_latest.sif"

# Calibrator sky models distributed with this repository
CAL_MODELS = {
    "CAS_A": CODE_ROOT / "CasA.skymodel",
    "CYG_A": CODE_ROOT / "CygA.skymodel",
    "VIR_A": CODE_ROOT / "VirA.skymodel",
}

if not CONTAINER_PATH.exists():
    print(f"[WARN] Missing container file: {CONTAINER_PATH}")

for k, p in CAL_MODELS.items():
    if not p.exists():
        print(f"[WARN] Missing skymodel for {k}: {p}")

# --------------------------------------------------
# Read selected plan and auto-pick calibrator info
# --------------------------------------------------
payload = json.load(open(PLAN_JSON, "r"))

CAL_SOURCE_REQUEST = payload.get("cal_source_request")
CAL_SOURCE_CHOSEN = payload.get("cal_source_chosen")
CAL_EVENT_CHOSEN = payload.get("cal_event_chosen")
CAL_CHOSEN_TAG = payload.get("cal_chosen_tag")

CAL_MODEL_PATH = CAL_MODELS.get(CAL_SOURCE_CHOSEN, None)

print("\\n=== Event context ===")
print("PLAN_JSON            =", PLAN_JSON)
print("EVENT_ROOT           =", EVENT_ROOT)
print("EVENT_NAME           =", EVENT_NAME)
print("EVENT_TAG            =", EVENT_TAG)

print("\\n=== Workflow roots ===")
print("STEP1_ROOT           =", STEP1_ROOT)
print("STEP2_ROOT           =", STEP2_ROOT)
print("STEP3_ROOT           =", STEP3_ROOT)
print("STEP4_ROOT           =", STEP4_ROOT)
print("IOCORR_ROOT          =", IOCORR_ROOT)
print("IOCENTROID_ROOT      =", IOCENTROID_ROOT)
print("BEAMPROP_ROOT        =", BEAMPROP_ROOT)

print("\\n=== Local resources ===")
print("CONTAINER_PATH       =", CONTAINER_PATH)

print("\\n=== Calibrator context ===")
print("CAL_SOURCE_REQUEST   =", CAL_SOURCE_REQUEST)
print("CAL_SOURCE_CHOSEN    =", CAL_SOURCE_CHOSEN)
print("CAL_CHOSEN_TAG       =", CAL_CHOSEN_TAG)
print("CAL_EVENT_CHOSEN     =", CAL_EVENT_CHOSEN)
print("CAL_MODEL_PATH       =", CAL_MODEL_PATH)

# Step-1 — Prepare Calibrator & Sun MS

## Goal
Prepare clean working copies of:
- CasA (calibrator)
- Sun (target)

No gain solving is done here.

---

## 1A — CasA: AOFlagger + Average

DP3:
steps=[flag,averager]
flag.type=aoflagger
averager.timestep=1
averager.freqstep=1

Output:
CasA_SBxxx_prep.MS

Purpose:
- Remove RFI
- Standardize resolution

---

## 1B — CasA: Preflag bad baselines (in-place)

DP3:
steps=[flag]
flag.type=preflagger
flag.baseline='MR102NEN&&*;MR103NEN&&*'

Purpose:
- Remove unstable baselines
- Stabilize calibration

---

## 1C — Sun: Clear flags → new copy

DP3:
steps=[flag]
flag.type=preflagger
flag.mode=clear
flag.baseline='*&&*'

Output:
SUN_SBxxx_prep.MS

Purpose:
- Reset flags
- Protect raw data
- Prepare for calibration

---

## Result per SB
CasA_SBxxx_prep.MS
SUN_SBxxx_prep.MS
(log files)

Step-1 = data hygiene stage.
Safe to re-run.

In [ ]:
run_step1_ui(
    PLAN_JSON,
    out_root=STEP1_ROOT,
)

## Step-2 — Zooming in (ROI Time Selection)

**Goal**  
Extract a time window (Region of Interest, ROI) from the prepared SUN MS produced in Step-1.

**Input**
- `SUN_{SB}_prep.MS` (from Step-1)
- Start time (e.g. `2024/03/10/12:00:00`)
- End time (e.g. `2024/03/10/12:20:00`)

**Operation (DP3)**
```bash
DP3 msin=SUN_{SB}_prep.MS \
     msin.starttime='YYYY/MM/DD/HH:MM:SS' \
     msin.endtime='YYYY/MM/DD/HH:MM:SS' \
     steps=[] \
     msout=SUN_{SB}_ROI.MS


**Output:**

SUN_{SB}_ROI.MS

**Located in:**

step2_outputs_YYYYMMDD/ROI/SBxxx/

In [ ]:
run_step2_zoom_ui(
    plan_json_path=PLAN_JSON,
    step1_root=STEP1_ROOT,
    out_root=STEP2_ROOT,
    default_start="2024/03/10/12:08:30",
    default_end="2024/03/10/12:08:50",
)

## Step-3 — DI Calibration (CasA → Sun ROI)

### Goal
Use the **CasA calibrator** to derive per-antenna gain solutions, then **apply** them to the **Sun ROI MS**, producing a calibrated data column (**CORR_NO_BEAM**).

> Note: **applybeam is intentionally NOT used** for NenuFAR solar imaging (cannot apply beam in this workflow).  
> We stop at: **gaincal (predict/solve) + applycal**.

---

### Inputs
For each selected subband `SBxxx.MS`:

1. **Calibrator MS (CasA)**  
   From **Step-1 outputs** (selected by `dropdown/pre/post`):
   - `step1_root/SBxxx/CasA_SBxxx_prep.MS`

2. **Target MS (Sun ROI)**  
   From **Step-2 outputs**:
   - `step2_root/ROI/SBxxx/SUN_SBxxx_ROI.MS`

3. **Calibrator sky model database**
   - `CasA.sourcedb` (pre-generated, provided by you)

---

### What it runs (per SB)
#### 3A) `gaincal` on CasA (solve gains)
- Generates a parset:
  - `step3_root/SBxxx/SBxxx_gaincal.parset`
- Runs DP3 gaincal, producing a solution table:
  - `step3_root/SBxxx/CasA_SBxxx_prep.MS/instrument`

#### 3B) `applycal` on Sun ROI (apply gains)
- Generates a parset:
  - `step3_root/SBxxx/SBxxx_applycal.parset`
- Runs DP3 applycal on:
  - `step2_root/ROI/SBxxx/SUN_SBxxx_ROI.MS`
- Writes calibrated visibilities into a **new data column**:
  - **`CORR_NO_BEAM`**  
  (in-place inside the same Sun ROI MS)

---

### Outputs
Per `SBxxx` you will get:

**Files (in Step-3 folder)**
- `step3_root/SBxxx/SBxxx_gaincal.parset`
- `step3_root/SBxxx/SBxxx_applycal.parset`
- logs:
  - `01_gaincal.log`
  - `02_applycal.log`

**Data product (in Step-2 ROI MS)**
- `step2_root/ROI/SBxxx/SUN_SBxxx_ROI.MS`
  - now contains a new column: **`CORR_NO_BEAM`**

---

### Why this matters
After Step-3, the Sun ROI dataset is calibrated (direction-independent, no beam) and ready for imaging steps that expect:
- stable gain-corrected visibilities
- `CORR_NO_BEAM` as the calibrated column

---

### Quick checks
**1) confirm CORR_NO_BEAM exists**
```bash
taql "select COLNAME() from /path/to/SUN_SBxxx_ROI.MS"

In [ ]:
run_step3_calib_ui(
    plan_json_path=PLAN_JSON,
    step1_root=STEP1_ROOT,
    step2_root=STEP2_ROOT,
    out_root=STEP3_ROOT,
    sourcedb=CAL_MODEL_PATH,
)

## Step4 — Imaging (WSClean) → FITS images

This step runs **WSClean** imaging on the ROI MS produced in Step2 (and calibrated products from Step3 if configured in the workflow).

**How to use**
1. Confirm your inputs:
   - `plan_json_path` points to your `selected_sb_pair_list.json`
   - `step2_root` points to `step2_outputs_YYYYMMDD`
   - `out_root` points to `step4_outputs_YYYYMMDD`
2. Run the cell to open the Step4 widget UI.
3. In the UI, select SB(s) and set WSClean imaging parameters:
   - image size / scale, weighting, robust, niter, auto-mask, etc.
4. Click **Run Step-4 (WSClean)**.

**Outputs**
- FITS images are written under:
  - `step4_outputs_YYYYMMDD/SBxxx/*-image.fits`
- WSClean logs are printed in the notebook and saved under the corresponding SB output folder.

In [ ]:
run_step4_wsclean_ui(
    plan_json_path=PLAN_JSON,
    step2_root=STEP2_ROOT,
    out_root=STEP4_ROOT,
)

## Step4 Quicklook — PNG quicklooks / optional movies from FITS

This tool reads Step4 FITS images (`*-image.fits`) and generates:
- quicklook PNG(s)
- optional movie (MP4 if `ffmpeg` exists; otherwise GIF fallback)

**How to use**
1. Set `step4_root` to your `step4_outputs_YYYYMMDD` folder.
2. Run the cell to open the widget UI.
3. In the UI:
   - choose the SB
   - multi-select one or more `*-image.fits`
   - set crop half-width (arcsec), CLim percentiles, contours
   - optionally enable **Make video** and set FPS
4. Click **Generate Quicklooks**.

**Outputs**
- PNGs:
  - `step4_outputs_YYYYMMDD/quicklook/SBxxx/*.png`
- Optional movie:
  - `step4_outputs_YYYYMMDD/quicklook/SBxxx/*quicklook.mp4` (or `.gif`)
- Log:
  - `step4_outputs_YYYYMMDD/quicklook/SBxxx/quicklook.log`

**Note**
Only `*-image.fits` are considered science images here (other FITS products are ignored).

In [ ]:
run_step4_quicklook_ui(
    step4_root=STEP4_ROOT,
    out_root=STEP4_ROOT / f"quicklook_{EVENT_TAG}",
)

## Post-processing tool A — WCS offset solve

This tool derives a WCS offset solution from a Quiet Sun frame.

### Purpose
It measures the Quiet Sun centroid in a selected FITS frame and estimates the offset relative to the expected solar-disc center.

### What this tool does
- loads a Quiet Sun FITS frame
- defines a measurement ROI
- measures the Quiet Sun centroid
- derives a WCS offset solution
- saves the solution and diagnostic quicklook

### How to use
1. Run this cell to open the widget UI.
2. Choose the SB and the Quiet Sun FITS frame.
3. Set the ROI around the Quiet Sun emission.
4. Click **Solve IO offset**.
5. Inspect the quicklook and saved solution.

### Inputs
- Step4 FITS products

### Outputs
- WCS offset solution file
- diagnostic quicklook
- log information

### Notes
- This is a **post-processing tool**, not part of the main Step1–Step4 imaging chain.
- The saved solution can be used by the next tool to correct selected FITS frames.

In [ ]:
run_iocorr_solve_ui(
    step4_root=STEP4_ROOT,
    out_root=IOCORR_ROOT,
    plan_json_path=PLAN_JSON,
)

## Post-processing tool B — WCS correction apply

This tool applies a previously derived WCS offset solution to selected FITS frames.

### Purpose
Once the Quiet Sun offset has been measured, the same WCS correction can be applied consistently to burst FITS products from the same SB.

### What this tool does
- loads a saved WCS offset solution
- applies the corresponding WCS correction to selected FITS frames
- writes corrected FITS products
- optionally generates before/after quicklooks and a movie

### How to use
1. Run the previous WCS offset solve tool first.
2. Run this cell to open the widget UI.
3. Choose the SB and select one or more FITS frames.
4. Click **Apply IO correction**.
5. Inspect the corrected FITS and before/after outputs.

### Inputs
- Step4 FITS products
- saved WCS offset solution

### Outputs
- corrected FITS products
- before/after quicklooks
- optional movie
- log file

### Notes
- This is a **post-processing tool**, not part of the main Step1–Step4 imaging chain.
- The corrected FITS products can be used directly by the centroid and beam-propagation tools.

In [ ]:
run_iocorr_apply_ui(
    step4_root=STEP4_ROOT,
    out_root=IOCORR_ROOT,
    plan_json_path=PLAN_JSON,
)

## Post-processing tool C — Centroid measurement

This tool measures radio-source centroid positions from selected FITS frames.

### Purpose
It provides a consistent way to measure source positions frame by frame, either from the original Step4 FITS products or from the WCS-corrected FITS products.

### What this tool does
- loads FITS frames from either:
  - Step4 raw products, or
  - WCS-corrected FITS products
- defines a measurement ROI
- fits the source centroid inside the ROI
- estimates centroid uncertainties
- optionally draws the beam ellipse
- saves centroid quicklooks and result tables

### How to use
1. Run this cell to open the widget UI.
2. Choose the source type and the SB.
3. Select one or more FITS frames.
4. Define the ROI and fit settings.
5. Click **Run centroid**.
6. Inspect the quicklooks and saved tables.

### Inputs
- Step4 FITS products
- optionally WCS-corrected FITS products

### Outputs
- centroid quicklook PNGs
- CSV / JSONL result tables
- log file
- optional movie

### Notes
- This is a **post-processing tool**, not part of the main Step1–Step4 imaging chain.
- The default input for positional analysis is usually the **WCS-corrected FITS** products.

In [ ]:
run_iocorr_centroid_ui(
    step4_root=STEP4_ROOT,
    step5b_root=IOCORR_ROOT,
    out_root=IOCENTROID_ROOT,
    plan_json_path=PLAN_JSON,
)

## Post-processing tool D — Beam propagation / projected source-height analysis

This tool converts measured radio-source positions into higher-level propagation diagnostics.

### Purpose
After imaging, WCS correction, and centroid measurement, this tool provides a geometry-based interpretation of the radio source evolution, especially for near-limb events.

### What this tool does
- loads FITS products from either:
  - Step4 raw products, or
  - WCS-corrected FITS products
- defines a measurement ROI and a display FOV
- measures source positions frame by frame
- supports two kinematics modes:
  - projected displacement
  - solar altitude
- supports either a blank background or an AIA background
- saves beam-propagation / source-tracking outputs for later analysis

### How to use
1. Run the previous imaging and post-processing steps first.
2. Run this cell to open the widget UI.
3. In the UI:
   - choose the input source
   - select one or more FITS frames
   - set the ROI and FOV
   - choose the kinematics mode
   - if needed, choose the projection mode and angle
   - click **Run**
4. Inspect the overplot and kinematics outputs.

### Inputs
- Step4 FITS products
- optionally WCS-corrected FITS products

### Outputs
- overplot figure
- kinematics figure
- CSV / JSONL tables
- summary and log files

### Notes
- This is a **post-processing analysis tool**, not part of the main Step1–Step4 imaging chain.
- For the current workflow, the default input is the **WCS-corrected FITS** products.
- In `projected displacement` mode, the projection settings are used.
- In `solar altitude` mode, the altitude is measured relative to the apparent solar limb.

In [ ]:
run_beam_propagation_ui(
    step4_root=STEP4_ROOT,
    step5b_root=IOCORR_ROOT,
    out_root=STEP6_ROOT,
    default_source="Step5B corrected",
    default_root=str(IOCORR_ROOT),
    default_roi=(0, 2000, -1000, 2000),          # measurement ROI
    default_fov=(-3000, 3000, -3000, 3000),      # display FOV
    default_thresh_frac=0.5,
    default_min_points=30,
    default_projection_mode="limb",
    default_projection_angle_deg=0.0,
    default_kinematics_mode="solar altitude",
    default_background_mode="blank",             # or "AIA"
    default_aia_wavelength=193,
    default_aia_time_mode="first",               # "first", "middle", or "manual"
    default_aia_time_manual="",
    default_jsoc_email="",
)